In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
df = spark.read.parquet("/user/hive/warehouse/sclsepb4532_raw.db/inventory", header=True, inferSchema=True)

26/11/05 18:12:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 18:12:56 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [2]:
df_cleaned1 = df.filter(F.col("quantity_available") > 0)

In [3]:
# Trims spaces and capitalizes the first letter (e.g., ' online ' becomes 'Online')
df_standardized = df_cleaned1.withColumn(
    "location_type", 
    F.initcap(F.trim(F.col("location_type")))
)

In [4]:
# Define your data retention window (e.g., 30 days)
retention_days = 30

# Keep only rows where the last_updated timestamp is within the retention period
df_fresh = df_standardized.filter(
    F.col("last_updated") >= F.date_sub(F.current_timestamp(), retention_days)
)

In [5]:
df_fresh.write.mode("overwrite").saveAsTable("sclsepb4532_staging.inventory_stg")

26/11/05 18:13:41 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
